# SteamSage — Run the ExaThon prototype in Google Colab

This notebook runs the SteamSage synthetic-data prototype without VS Code.

**Important:** The first part runs in Colab using the bundled synthetic CSV. To demonstrate the competition requirement, you must also connect to an Exasol Personal instance. A Colab runtime can connect only if your Exasol host/port is reachable from the internet or through an approved network tunnel. A database running only on your own laptop's `localhost` is **not** reachable from Colab.

Do not paste database passwords into notebook cells or share a notebook containing secrets. Use Colab Secrets for credentials if you configure Exasol access.


## 1. Upload the project ZIP

In [ ]:
from google.colab import files
uploaded = files.upload()  # Choose SteamSage_ExaThon_Project.zip


In [ ]:
import zipfile, os
from pathlib import Path

zip_name = next((name for name in uploaded if name.lower().endswith(".zip")), None)
if not zip_name:
    raise FileNotFoundError("Upload the SteamSage project ZIP first.")

extract_dir = Path("/content/SteamSage_ExaThon_Submission")
extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(extract_dir.parent)

# The ZIP contains a top-level SteamSage_ExaThon_Submission folder.
PROJECT = extract_dir
print("Project folder:", PROJECT)
print("Files:", [p.name for p in PROJECT.iterdir()])


## 2. Install the Python packages

In [ ]:
%pip -q install pandas streamlit python-dotenv pyexasol matplotlib


## 3. Load the synthetic steam-trap dataset

In [ ]:
import pandas as pd

DATA_PATH = PROJECT / "data" / "synthetic_steam_traps.csv"
df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
display(df.head(10))


## 4. Run the SteamSage diagnostic rules

In [ ]:
def diagnose(data):
    out = data.copy()
    out["diagnosis"] = "No immediate flag"
    out["priority"] = "Low"

    suspected_leak = (out["ultrasonic_index"] >= 65) & (out["temperature_c"] >= 145)
    suspected_blockage = (out["temperature_c"] <= 105) & (out["ultrasonic_index"] <= 40)
    uncertain = (~suspected_leak) & (~suspected_blockage) & (
        (out["ultrasonic_index"] >= 35) | (out["inspection_age_days"] >= 120)
    )

    out.loc[suspected_leak, "diagnosis"] = "Suspected leak"
    out.loc[suspected_blockage, "diagnosis"] = "Suspected blockage"
    out.loc[uncertain, "diagnosis"] = "Inspect / uncertain"

    out.loc[suspected_leak | suspected_blockage, "priority"] = "High"
    out.loc[uncertain, "priority"] = "Medium"

    out["estimated_annual_cost_inr"] = (
        out["estimated_loss_kg_hr"] / 1000
        * out["steam_cost_inr_per_tonne"]
        * out["operating_hours_per_day"] * 365
    ).round(0)

    priority_order = {"High": 0, "Medium": 1, "Low": 2}
    out["_sort"] = out["priority"].map(priority_order)
    return out.sort_values(
        ["_sort", "estimated_annual_cost_inr"], ascending=[True, False]
    ).drop(columns="_sort")

results = diagnose(df)
display(results.head(20))


## 5. View the dashboard metrics

In [ ]:
from IPython.display import display, Markdown

traps = len(results)
flagged = int((results["diagnosis"] != "No immediate flag").sum())
high_priority = int((results["priority"] == "High").sum())
annual_cost = results["estimated_annual_cost_inr"].sum()

display(Markdown(
    f"""
### SteamSage — Synthetic Demo
| Metric | Value |
|---|---:|
| Traps analyzed | {traps:,} |
| Flagged for review | {flagged:,} |
| High-priority checks | {high_priority:,} |
| Illustrative annual cost exposure | ₹{annual_cost:,.0f} |
"""
))


In [ ]:
import matplotlib.pyplot as plt

results["diagnosis"].value_counts().plot(kind="bar", title="SteamSage diagnosis mix")
plt.xlabel("Diagnosis")
plt.ylabel("Number of traps")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


## 6. Inspect one trap and export the results

In [ ]:
trap_id = "ST-002"  # Change this to any trap ID, e.g. ST-120
selected = results.loc[results["trap_id"] == trap_id]
display(selected)


In [ ]:
output_path = "/content/steamsage_diagnosis.csv"
results.to_csv(output_path, index=False)

from google.colab import files
files.download(output_path)


## 7. Exasol Personal connection — optional next step

The notebook above is the **local synthetic-data demo**, not yet proof of an Exasol-backed submission.

To use Exasol from Colab:
1. Start your Exasol Personal instance and obtain its connection host, port, username, and password.
2. Confirm the database endpoint is reachable from outside your laptop/network. `localhost` will not work from Colab.
3. Store credentials in Colab **Secrets** (key icon in the left sidebar), not in a shared cell.
4. Use `pyexasol.connect(...)` with those credentials, create the target table, upload the records, and run the SQL query in `sql/diagnosis.sql`.
5. Capture evidence showing the table and query running in Exasol Personal.

If Exasol Personal is running only on your laptop, use a supported secure network route or run the database in an accessible environment. Do not expose a database port publicly without proper access controls.

**Engineering disclaimer:** The dataset, labels, diagnostic thresholds, loss rates, and steam prices are synthetic or illustrative. They do not validate real-world fault-detection accuracy or guaranteed savings. Suspected faults must be checked by qualified personnel.
